# Access filtered LSST alert data via Pub/Sub

Messages published to the Pitt-Google Alert Broker's alert and value-added streams are publicly accessible to users via [Google Pub/Sub](https://docs.cloud.google.com/pubsub/docs). The names of the available topics are outlined in our [data listings page](https://mwvgroup.github.io/pittgoogle-client/listings.html#pub-sub-alert-streams).

## Table of Contents

- [Create a DIA object watchlist](#create-a-dia-object-watchlist)
- [Create a filter using HEALPix pixels](#create-a-filter-using-healpix-pixels)

## Prerequisites

- Complete the [One-Time Setup](https://mwvgroup.github.io/pittgoogle-client/one-time-setup/index.html), specifically:
     - Install the pittgoogle-client package
     - Setup authentication to a Google Cloud project
     - Set environment variables
     - Enable the Pub/Sub API
     - You can skip the command-line tools

In [1]:
import pittgoogle
import hpgeom

## Create a DIA object watchlist

A “[DIA object](https://dp1.lsst.io/products/catalogs/dia_object.html#description)” is an astrophysical transient or variable object at a static sky coordinate. The `diaObjectId` is the unique identifier of a `diaObject`. 

This `diaObjectId` can be used to filter messages in the Pitt-Google Alert Broker's alert and value-added streams. The result is a Pub/Sub subscription that will receive messages associated with detections of the `diaObject` or `diaObjects`.

In [2]:
# define the attribute filter for multiple DIA objects
dia_object_list = [
    170028485835751490,
    313853517494222904,
    313853517493174363,
] # add or remove diaObjectIds as necessary

_attribute_filter = " OR ".join(
    f'attributes.diaObject_diaObjectId = "{dia_object}"' for dia_object in dia_object_list
)

# inspect the attribute filter
_attribute_filter

'attributes.diaObject_diaObjectId = "170028485835751490" OR attributes.diaObject_diaObjectId = "313853517494222904" OR attributes.diaObject_diaObjectId = "313853517493174363"'

NOTE: The maximum length of a filter is 256 bytes. If your filter exceeds this limit, consider using a JavaScript User Defined Function (UDF) to filter alerts instead.

## Create subscription

In [3]:
# define the Pitt-Google Alert Broker topic that your subscription will subscribe to
topic = pittgoogle.Topic(name="lsst-alerts", projectid="pitt-alert-broker")

# subscription will only acknowledge messages that pass the attribute filter
subscription = pittgoogle.Subscription(
    name="dia-object-watchlist", # UPDATE ME -> name for your subscription
    topic=topic,
    schema_name="lsst",
)

# create the subscription
subscription.touch(attribute_filter=_attribute_filter)

## Create a filter using HEALPix pixels

It is a reasonable request to ask to receive alerts on a specific region (or regions) of the sky within a defined radius. The `pittgoogle-client` package exposes two mechanisms for this kind of alert filtering when creating a subscription to one of the Pitt-Google Alert Broker's Pub/Sub topics:

1. Attribute-based filters: lightweight filters that match on Pub/Sub message attributes
2. JavaScript User Defined Function (UDF) filters: more powerful filters that operate on the full message (payload and/or attributes)

The Pitt-Google Alert Broker includes the following keys as attributes to every single Pub/Sub message: `healpix9`, `healpix19`, and `healpix29`. The cell below demonstrates how to determine the HEALPix pixels (order 9) for a RA & Dec with a defined search cone radius, and use these pixels to filter Pub/Sub message based on their `healpix9` attribute value. You can update the values of these variables as necessary.

In [4]:
# define RA, Dec in degrees for the center of the search cone
coord = (61.608431, -48.709264) # UPDATE ME
radius = 1 / 3600 # deg (UPDATE ME)

# define an array containing a list of HEALPix pixels at a given order that overlap with the search cone
order = 9 # HEALPix order (UPDATE ME)
nside = hpgeom.order_to_nside(order)
cone = hpgeom.query_circle(nside, *coord, radius, inclusive=True)

# define the attribute filter
_attribute_filter = " OR ".join(
    f'attributes.healpix{order} = "{pixel}"'
    for pixel in cone
)

# inspect the attribute filter
_attribute_filter

'attributes.healpix9 = "2196702"'

## Create subscription

In [5]:
# define the Pitt-Google Alert Broker topic that your subscription will subscribe to
topic = pittgoogle.Topic(name="lsst-alerts", projectid="pitt-alert-broker")

# subscription will only acknowledge messages that pass the attribute filter
subscription = pittgoogle.Subscription(
    name=f"healpix{order}-watchlist", # UPDATE ME -> name for your subscription
    topic=topic,
    schema_name="lsst",
)

# create the subscription
subscription.touch(attribute_filter=_attribute_filter)

NOTE: A long list of HEALPix pixels will likely exceed the 256 bytes limit for attribute filters. If this is the case for your filter, we recommend using JavaScript user defined functions. We explicity demonstrate how to do this in the cells below.

In [6]:
# define RA, Dec in degrees for the center of the search cone
coord = (61.608431, -48.709264) # UPDATE ME
radius = 1 / 3600 # deg (UPDATE ME)

# define an array containing a list of HEALPix pixels at a given order that overlap with the search cone
order = 19 # HEALPix order (UPDATE ME)
nside = hpgeom.order_to_nside(order)
cone = hpgeom.query_circle(nside, *coord, radius, inclusive=True)

# inspect the list of HEALPix pixels
cone

array([2303409453627, 2303409453629, 2303409453630, 2303409453631,
       2303409453672, 2303409453673, 2303409453674, 2303409453675,
       2303409453678, 2303409453703, 2303409453709, 2303409453712,
       2303409453713, 2303409453714, 2303409453715, 2303409453716,
       2303409453717, 2303409453718, 2303409453719, 2303409453720,
       2303409453721, 2303409453722, 2303409453723, 2303409453724,
       2303409453725, 2303409453726, 2303409453727, 2303409453760,
       2303409453761, 2303409453762, 2303409453763, 2303409453764,
       2303409453768, 2303409453769])

## Define JavaScript UDF and create subscription
We use the list of HEALPix pixels shown above to define the set of pixels we would like to incorporate into our filter.

In [ ]:
# define the JavaScript UDF to filter messages
_smt_javascript_udf = f'''
function filterByHEALPix(message, metadata) {{
    const CONE_PIXELS = new Set({cone.tolist()});
    const attrs = message.attributes || {{}};
    const healpix{order} = attrs.healpix{order} ? parseInt(attrs.healpix{order}) : null;
    return (healpix{order} !== null && CONE_PIXELS.has(healpix{order})) ? message : null;
}}
'''

# subscription will only acknowledge messages that pass the attribute filter
subscription = pittgoogle.Subscription(
    name=f"healpix{order}-watchlist", # UPDATE ME -> name for your subscription
    topic=topic,
    schema_name="lsst",
)

# create the subscription
subscription.touch(smt_javascript_udf=_smt_javascript_udf)